<a href="https://colab.research.google.com/github/Shamsfathalla/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shamsfathalla/FlyRank-Starter-Notebooks/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule and its reason codes

My rule finds pages with high views but zero clicks.

Score: gsc_impressions minus gsc_clicks.

Reason code: NEEDS_TITLE_REFRESH.

Search volume vs zero clicks. Verdict: CONFIRMED.

Average position vs CTR. Verdict: CONFIRMED.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata

# Load token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Connect DuckDB to Hugging Face
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
table_path = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"

print("Warehouse connection verified and ready.")

Warehouse connection verified and ready.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Signal 1: Volume vs Clicks
print("Signal 1: Volume bucket vs Clicks")
print(con.sql(f"""
    SELECT
        CASE WHEN gsc_impressions > 500 THEN 'High' ELSE 'Low' END as vol_bucket,
        COUNT(*) as n,
        AVG(gsc_clicks) as avg_clicks
    FROM {table_path}
    WHERE month = '2026-03'
    GROUP BY 1
""").df())

# Signal 2: Position vs CTR
print("\nSignal 2: Position bucket vs CTR")
print(con.sql(f"""
    SELECT
        CASE WHEN gsc_avg_position <= 10 THEN 'Page 1' ELSE 'Page 2+' END as pos_bucket,
        COUNT(*) as n,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as avg_ctr
    FROM {table_path}
    WHERE month = '2026-03' AND gsc_impressions > 10
    GROUP BY 1
""").df())

Signal 1: Volume bucket vs Clicks


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  vol_bucket        n  avg_clicks
0        Low  9740242    0.053827
1       High   101136    2.942019

Signal 2: Position bucket vs CTR


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  pos_bucket        n   avg_ctr
0     Page 1  1309027  0.003356
1    Page 2+   770400  0.001903


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The code calculates the score, assigns the NEEDS_TITLE_REFRESH reason code and the REVIEW action label, then saves the sorted list to a CSV.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

q = f"""
    SELECT
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        (gsc_impressions - gsc_clicks) AS score,
        'NEEDS_TITLE_REFRESH' AS reason_code,
        'REVIEW' AS action_label
    FROM {table_path}
    WHERE month = '2026-03' AND gsc_impressions > 100
    ORDER BY score DESC
"""
df = con.sql(q).df()

os.makedirs("work/outputs", exist_ok=True)
df.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Saved queue to CSV. Rows:", len(df))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved queue to CSV. Rows: 633483


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For all top 20 rows:

Action: REVIEW.

Reason: High impressions but missed clicks.

Confidence: High, based on raw metrics.

What makes it wrong: The search was a zero-click query where the user read the answer directly on the Google search results page so they didn't need to click our link.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(df.head(20)[['content_hash_id', 'score', 'reason_code', 'action_label']])

             content_hash_id  score          reason_code action_label
0   content_44f34c0a90047651  40083  NEEDS_TITLE_REFRESH       REVIEW
1   content_eadb33b5df496f4a  39053  NEEDS_TITLE_REFRESH       REVIEW
2   content_34a70fea29d15f24  39001  NEEDS_TITLE_REFRESH       REVIEW
3   content_eadb33b5df496f4a  38165  NEEDS_TITLE_REFRESH       REVIEW
4   content_945d6ff91386c817  37368  NEEDS_TITLE_REFRESH       REVIEW
5   content_eadb33b5df496f4a  35179  NEEDS_TITLE_REFRESH       REVIEW
6   content_eadb33b5df496f4a  34594  NEEDS_TITLE_REFRESH       REVIEW
7   content_eadb33b5df496f4a  34371  NEEDS_TITLE_REFRESH       REVIEW
8   content_fec55986a1868d62  33383  NEEDS_TITLE_REFRESH       REVIEW
9   content_eadb33b5df496f4a  33356  NEEDS_TITLE_REFRESH       REVIEW
10  content_44f34c0a90047651  32958  NEEDS_TITLE_REFRESH       REVIEW
11  content_44f34c0a90047651  32754  NEEDS_TITLE_REFRESH       REVIEW
12  content_eadb33b5df496f4a  32444  NEEDS_TITLE_REFRESH       REVIEW
13  content_eadb33b5

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Pages stuck at the bottom of page 1 (positions 8-10) naturally get low clicks.

Leakage check: Verified dates do not pass March 2026 and no hidden flags were used.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
max_date = df['report_date'].max()
print("Max date:", max_date)

max_date_str = str(max_date)[:10]
assert max_date_str <= '2026-03-31', "Future data leaked!"

bad_cols = [c for c in df.columns if 'flag' in c or 'opportunity' in c]
assert len(bad_cols) == 0, "Leaked labels used!"
print("Leakage checks passed.")

Max date: 2026-03-31 00:00:00
Leakage checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.